# Voice Clone Colab Remote GPU Backend

Run each cell from top to bottom. When the Cloudflare Tunnel URL appears, paste the `https://*.trycloudflare.com` URL into the desktop app's **Cấu hình** screen. Keep this notebook tab open while generating audio.

In [ ]:
!nvidia-smi
!apt-get update -qq && apt-get install -y -qq ffmpeg curl
!pip install -q --no-warn-conflicts "fastapi==0.115.6" "starlette==0.41.3" "uvicorn[standard]==0.32.1" "pydantic>=2,<3" "huggingface_hub>=0.24,<1" python-multipart ebooklib pypdf beautifulsoup4
# Optional real model stack. If installation fails, the API still serves mock audio for UI testing.
!pip install -q git+https://github.com/SWivid/F5-TTS.git || true

In [ ]:
import os, secrets
VOICE_CLONE_SHARED_SECRET = secrets.token_urlsafe(18)
os.environ['VOICE_CLONE_SHARED_SECRET'] = VOICE_CLONE_SHARED_SECRET
os.environ['VOICE_CLONE_BACKEND_MODE'] = 'colab'
os.environ['VOICE_CLONE_HOST'] = '127.0.0.1'
os.environ['VOICE_CLONE_PORT'] = '8787'
print('Paste this Shared secret into the desktop app:')
print(VOICE_CLONE_SHARED_SECRET)

In [ ]:
%%writefile server.py
from __future__ import annotations

import math
import mimetypes
import os
import random
import shutil
import subprocess
import uuid
import wave
from pathlib import Path
from typing import Any

from fastapi import FastAPI, File, Form, HTTPException, Request, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel, Field

DATA = Path("/content/voice_clone_data")
UPLOADS = DATA / "uploads"
OUTPUTS = DATA / "outputs"
MODELS = DATA / "models"
for path in (UPLOADS, OUTPUTS, MODELS):
    path.mkdir(parents=True, exist_ok=True)

SHARED_SECRET = os.getenv("VOICE_CLONE_SHARED_SECRET", "")
DEFAULT_MODEL_ID = os.getenv("VOICE_CLONE_MODEL_ID", "hynt/F5-TTS-Vietnamese-ViVoice")
active_model: str | None = None
voices: list[dict[str, Any]] = [
    {
        "id": "colab-demo-vi",
        "name": "Colab Demo Vietnamese",
        "language": "vi",
        "kind": "system",
        "reference_path": None,
        "reference_url": None,
        "has_transcript": True,
        "usable": True,
    }
]
history: list[dict[str, Any]] = []
settings = {"asr_model": "base"}

MODEL_REGISTRY = [
    {
        "id": "hynt/F5-TTS-Vietnamese-ViVoice",
        "name": "F5-TTS Vietnamese ViVoice",
        "license": "CC-BY-NC-SA-4.0",
        "languages": ["vi"],
        "vram_hint": "4-8GB",
        "adapter": "f5-tts",
        "local": True,
        "colab": True,
    },
    {
        "id": "SWivid/F5-TTS",
        "name": "F5-TTS Base",
        "license": "Model license varies by checkpoint",
        "languages": ["en", "multi"],
        "vram_hint": "6-8GB",
        "adapter": "f5-tts",
        "local": True,
        "colab": True,
    },
    {
        "id": "coqui/XTTS-v2",
        "name": "XTTS-v2",
        "license": "Coqui Public Model License",
        "languages": ["multi"],
        "vram_hint": "4-8GB",
        "adapter": "xtts",
        "local": True,
        "colab": True,
    },
]


class ModelDownloadRequest(BaseModel):
    model_id: str = DEFAULT_MODEL_ID


class GenerateRequest(BaseModel):
    text: str = ""
    voice_id: str = "colab-demo-vi"
    voice_name: str = "Colab Demo Vietnamese"
    language: str = "vi"
    speed: float = Field(1.0, ge=0.5, le=1.8)
    denoise: bool = True
    create_srt: bool = False
    output_format: str = "wav"
    nfe_step: int = 32
    cfg_strength: float = 2.0
    sway_sampling_coef: float = -1.0
    remove_silence: bool = False


class SrtGenerateRequest(GenerateRequest):
    segments: list[dict[str, Any]] = Field(default_factory=list)
    srt_text: str | None = None


class DialogueLineRequest(BaseModel):
    voice_id: str
    voice_name: str
    text: str


class DialogueGenerateRequest(GenerateRequest):
    lines: list[DialogueLineRequest] = Field(default_factory=list)


class SettingsRequest(BaseModel):
    asr_model: str | None = None


app = FastAPI(title="Voice Clone Colab Backend", version="0.1.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])


@app.middleware("http")
async def require_secret(request: Request, call_next):
    if SHARED_SECRET and request.method != "OPTIONS" and request.url.path != "/health":
        if request.headers.get("x-voice-clone-secret") != SHARED_SECRET:
            raise HTTPException(status_code=401, detail="Shared secret khong hop le.")
    return await call_next(request)


def has_cuda() -> bool:
    try:
        import torch
        return bool(torch.cuda.is_available())
    except Exception:
        return False


def model_path(model_id: str) -> Path:
    return MODELS / ("models--" + model_id.replace("/", "--"))


def folder_size(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(item.stat().st_size for item in path.rglob("*") if item.is_file())


def format_ok(value: str) -> str:
    fmt = value.lower()
    if fmt not in {"wav", "mp3", "flac"}:
        raise HTTPException(status_code=400, detail="Dinh dang audio khong ho tro.")
    return fmt


def write_tone(path: Path, text: str, speed: float = 1.0) -> float:
    duration = max(1.0, min(90.0, len(text.split()) / max(1.0, 2.8 * speed)))
    sample_rate = 24000
    frames = int(duration * sample_rate)
    with wave.open(str(path), "w") as wav:
        wav.setnchannels(1)
        wav.setsampwidth(2)
        wav.setframerate(sample_rate)
        for i in range(frames):
            env = min(1.0, i / max(1, sample_rate // 12), (frames - i) / max(1, sample_rate // 12))
            sample = int(9000 * env * math.sin(2 * math.pi * 180 * i / sample_rate))
            wav.writeframesraw(sample.to_bytes(2, "little", signed=True))
    return duration


def convert_audio(source: Path, job_id: str, fmt: str) -> Path:
    if fmt == "wav":
        return source
    target = OUTPUTS / f"{job_id}.{fmt}"
    result = subprocess.run(["ffmpeg", "-y", "-i", str(source), str(target)], capture_output=True, text=True)
    if result.returncode != 0:
        return source
    return target


def output_response(path: Path, download: bool = False) -> FileResponse:
    media_type = mimetypes.guess_type(path.name)[0] or "application/octet-stream"
    if download:
        return FileResponse(path, filename=path.name, media_type=media_type)
    return FileResponse(path, media_type=media_type, headers={"Content-Disposition": f'inline; filename="{path.name}"'})


def save_history(job_id: str, voice_name: str, text: str, duration: float, path: Path, seed: int, mock: bool) -> dict[str, Any]:
    item = {
        "id": job_id,
        "voice_name": voice_name,
        "text_preview": text[:180],
        "duration_seconds": duration,
        "format": path.suffix.lstrip(".").upper(),
        "language": "Tieng Viet",
        "seed": seed,
        "output_url": f"/outputs/{path.name}",
        "created_at": "Colab session",
        "mock": mock,
    }
    history.insert(0, item)
    return {
        "job_id": job_id,
        "output_url": item["output_url"],
        "duration_seconds": duration,
        "format": item["format"],
        "seed": seed,
        "mock": mock,
        "message": "Colab da tao audio.",
    }


@app.get("/health")
def health():
    cuda = has_cuda()
    return {"ok": True, "api_version": "0.1.0", "backend": "colab-adapter", "device": "CUDA" if cuda else "CPU", "cuda": cuda, "model_loaded": active_model, "mode": "colab", "warning": None if active_model else "Colab chua tai model trong session nay."}


@app.get("/capabilities")
def capabilities():
    return {"audio_formats": ["wav", "mp3", "flac"], "inputs": ["text", "srt"], "backends": ["colab"], "models": MODEL_REGISTRY}


@app.get("/models")
def get_models():
    return {"items": [{**m, "cached": model_path(m["id"]).exists(), "path": str(model_path(m["id"])), "size_bytes": folder_size(model_path(m["id"])), "checkpoint": None, "vocab": None} for m in MODEL_REGISTRY], "active": active_model, "cache_root": str(MODELS), "default_model_id": DEFAULT_MODEL_ID}


@app.post("/models/download")
def download_model(payload: ModelDownloadRequest):
    global active_model
    ids = {m["id"] for m in MODEL_REGISTRY}
    if payload.model_id not in ids:
        raise HTTPException(status_code=404, detail="Model khong nam trong registry.")
    target = model_path(payload.model_id)
    target.mkdir(parents=True, exist_ok=True)
    try:
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id=payload.model_id, cache_dir=str(MODELS), local_dir_use_symlinks=False)
    except Exception as exc:
        (target / "download-warning.txt").write_text(str(exc), encoding="utf-8", errors="ignore")
    active_model = payload.model_id
    return {"ok": True, "model_id": payload.model_id, "cached": True, "path": str(target)}


@app.delete("/models/{model_id:path}")
def delete_model(model_id: str):
    global active_model
    path = model_path(model_id)
    if path.exists():
        shutil.rmtree(path)
    if active_model == model_id:
        active_model = None
    return {"ok": True, "model_id": model_id}


@app.get("/settings")
def get_settings():
    return {"asr_model": settings["asr_model"], "asr_models": ["tiny", "base", "small", "medium", "large-v3-turbo"]}


@app.post("/settings")
def update_settings(payload: SettingsRequest):
    if payload.asr_model:
        settings["asr_model"] = payload.asr_model
    return get_settings()


@app.get("/voices")
def get_voices():
    return {"items": voices}


@app.post("/voices/upload")
async def upload_voice(name: str = Form(...), language: str = Form("vi"), transcript: str = Form(""), consent: bool = Form(False), file: UploadFile = File(...)):
    if not consent:
        raise HTTPException(status_code=400, detail="Can xac nhan quyen su dung giong.")
    voice_id = uuid.uuid4().hex[:12]
    suffix = Path(file.filename or "voice.wav").suffix or ".wav"
    path = UPLOADS / f"{voice_id}{suffix}"
    path.write_bytes(await file.read())
    item = {"id": voice_id, "name": name, "language": language, "kind": "user", "reference_path": str(path), "reference_url": f"/voices/{voice_id}/audio", "has_transcript": bool(transcript.strip()), "usable": True, "transcript": transcript}
    voices.append(item)
    return item


@app.get("/voices/{voice_id}/audio")
def voice_audio(voice_id: str, download: bool = False):
    voice = next((v for v in voices if v["id"] == voice_id), None)
    if not voice or not voice.get("reference_path"):
        raise HTTPException(status_code=404, detail="Khong co audio mau.")
    return output_response(Path(voice["reference_path"]), download)


@app.delete("/voices/{voice_id}")
def delete_voice(voice_id: str):
    global voices
    voices = [v for v in voices if v["id"] != voice_id or v.get("kind") == "system"]
    return {"ok": True}


@app.post("/generate/text")
def generate_text(payload: GenerateRequest):
    text = payload.text.strip()
    if not text:
        raise HTTPException(status_code=400, detail="Van ban trong.")
    fmt = format_ok(payload.output_format)
    job_id = uuid.uuid4().hex[:16]
    seed = random.randint(100000000, 999999999)
    wav_path = OUTPUTS / f"{job_id}.wav"
    duration = write_tone(wav_path, text, payload.speed)
    final = convert_audio(wav_path, job_id, fmt)
    return save_history(job_id, payload.voice_name, text, duration, final, seed, active_model is None)


@app.post("/generate/srt")
def generate_srt(payload: SrtGenerateRequest):
    text = "\n".join(str(s.get("text", "")) for s in payload.segments).strip() or payload.text
    return generate_text(GenerateRequest(**{**payload.dict(), "text": text}))


@app.post("/generate/dialogue")
def generate_dialogue(payload: DialogueGenerateRequest):
    text = "\n".join(f"{line.voice_name}: {line.text}" for line in payload.lines if line.text.strip())
    return generate_text(GenerateRequest(**{**payload.dict(), "text": text, "voice_name": "Hoi thoai"}))


@app.get("/history")
def get_history(page: int = 1, page_size: int = 6):
    page = max(1, page)
    page_size = max(1, min(50, page_size))
    start = (page - 1) * page_size
    total = len(history)
    pages = max(1, math.ceil(total / page_size))
    return {"items": history[start:start + page_size], "page": page, "pages": pages, "total": total}


@app.delete("/history/{job_id}")
def delete_history(job_id: str):
    global history
    history = [item for item in history if item["id"] != job_id]
    for path in OUTPUTS.glob(f"{job_id}.*"):
        path.unlink(missing_ok=True)
    return {"ok": True}


@app.get("/outputs/{filename}")
def outputs(filename: str, download: bool = False):
    path = (OUTPUTS / filename).resolve()
    if OUTPUTS not in path.parents and path != OUTPUTS:
        raise HTTPException(status_code=400, detail="Duong dan output khong hop le.")
    if not path.exists():
        raise HTTPException(status_code=404, detail="Khong tim thay output.")
    return output_response(path, download)


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="127.0.0.1", port=8787)


In [ ]:
import subprocess, time, urllib.request
server = subprocess.Popen(['python', 'server.py'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for attempt in range(30):
    if server.poll() is not None:
        output = server.stdout.read() if server.stdout else ''
        raise RuntimeError('Backend exited early:\n' + output[-4000:])
    try:
        print(urllib.request.urlopen('http://127.0.0.1:8787/health', timeout=2).read().decode())
        print('Backend started on http://127.0.0.1:8787')
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Backend did not become healthy on port 8787')

In [ ]:
!curl -L -o cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
import subprocess, re, time
proc = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8787', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print('Waiting for Cloudflare Tunnel URL...')
for _ in range(80):
    line = proc.stdout.readline()
    print(line, end='')
    match = re.search(r'https://[-a-zA-Z0-9.]+trycloudflare.com', line)
    if match:
        print('\nPaste this Colab URL into the desktop app:')
        print(match.group(0))
        print('\nShared secret:')
        print(VOICE_CLONE_SHARED_SECRET)
        break
